## -----------------------------------------------------------------------------
## INITIALIZATION
## -----------------------------------------------------------------------------
useful lines:

with_rx.printSchema()

train_df.select("ENROLID").distinct().count()

In [1]:
import sys
print(f"Python version: {sys.version}")
import json
import logging
import csv
import gzip
import re
import pandas as pd
import numpy as np
from functools import reduce
from pyspark.sql.types import StringType,DecimalType,DoubleType,IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler, Bucketizer
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession,Row
from pyspark import SparkConf
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# -----------------------------------------------------------------------------
# INITIALIZE LOGGING
# -----------------------------------------------------------------------------
f = '%(asctime)-15s %(levelname)-8s %(message)s'
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")
logging.basicConfig(format=f)


from IPython.core.magic import register_cell_magic

# -----------------------------------------------------------------------------
# start_spark
# -----------------------------------------------------------------------------
def start_spark(
    driver_memory="100g",
    storage_fraction=0.5,
    num_nodes=10,
):
    """Initialize spark

    Arguments:
        driver_memory: Maximum heap size for the Spark driver Java
            virtual machine.
        storage_fraction: Controls what portion of Spark's unified
            memory is reserved for storage (i.e., caching/persisting data
            and broadcast variables), as a fraction of the total
            execution + storage memory pool.
            If you cache/persist a lot of data, and you're evicting
            data too early, you might increase this value (e.g. 0.6 or 0.7).
            Conversely, if your job is shuffle-heavy and fails due to
            memory pressure, you might decrease it (e.g. 0.3).
        num_nodes: How many concurrent threads to use while running
            in "local mode" (i.e. in a single machine instead of a cluster).
            Use '*' to use all cores, or an integer > 0 for a specific
            number of threads.
    """

    conf = SparkConf().setAppName("My_Application")
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.memory.storageFraction", str(storage_fraction))
    conf.setMaster(f"local[{num_nodes}]")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel('WARN')

    return spark


Python version: 3.11.0 (main, Jun 13 2025, 14:48:45) [Clang 16.0.0 (clang-1600.0.26.6)]


In [2]:

spark = start_spark(num_nodes=10)
#spark.stop()

  
@register_cell_magic
def spark_sql(line, cell):
    result = spark.sql(cell)
    result.show(n=1000)
  

# -- READ ENROLLMENT AND DATA TABLES
enrollment_file = f"/Users/Charles/DATA/ckd/ckd_enrollment"
logger.info(f">>> Reading enrollment file: {enrollment_file}")
df_enrollment = spark.read.format("parquet").load(enrollment_file)
df_enrollment.createOrReplaceTempView('enrollment')
logger.info(f">>> ENROLLMENT has {df_enrollment.count():,} rows")
logger.info(f">>> ENROLLMENT has {df_enrollment.select('ENROLID').distinct().count():,} unique enrollees")

claims_file = f"/Users/Charles/DATA/ckd/ckd_claims"
logger.info(f">>> Reading claims file: {claims_file}")
df_claims = spark.read.format("parquet").load(claims_file)
df_claims.createOrReplaceTempView('claims')
logger.info(f">>> CLAIMS has {df_claims.count():,} rows")
logger.info(f">>> CLAIMS has {df_claims.select('ENROLID').distinct().count():,} unique enrollees")

# -- NOTE: with the "createOrReplaceTempView" we define a view of these
# -- tables, so we can use them in SQL queries.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/17 15:03:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/17 15:03:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2025-08-17 15:03:56,664 INFO     >>> Reading enrollment file: /Users/Charles/DATA/ckd/ckd_enrollment
25/08/17 15:03:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2025-08-17 15:03:58,401 INFO     >>> ENROLLMENT has 2,586,930 rows
2025-08-17 15:04:00,228 INFO     >>> ENROLLMENT has 862,310 unique enrollees    
2025-08-17 15:04:00,229 INFO     >>> Reading claims file: /Users/Charles/DATA/ckd/ckd_claims
2025-08-17 15:04

## Load Pilot IDs

In [ ]:
# -----------------------------------------------------------------------------
# CKD COST FEATURE ENGINEERING FOR 2018 PREDICTION MODEL
# -----------------------------------------------------------------------------
# 
# Context: This code creates CKD-related cost features from 2017 claims data
# for predicting 2018 high-cost outcomes in an existing pilot cohort.
# 
# Input: pilot_claims (claims data already filtered to pilot cohort IDs)
# Output: CKD cost features for 2017 by quarter and annually
# Target: 2018 cost category prediction
# 
# -----------------------------------------------------------------------------


In [ ]:
pilot_ids =spark.read.format("parquet").load( "df_0714_2017_18_with_THERCLS.parquet")

pilot_claims = (
    df_claims
      .join(pilot_ids, on="ENROLID", how="inner")
      .orderBy("ENROLID","SVCDATE")
)

In [12]:
pilot_claims.count()


16442410

# Cost aggregation for only CKD-related costs
might not need to ECDF by region, just cost magnitude and focus on gradient computation.
inflation is still necessary

In [4]:
# -----------------------------------------------------------------------------
# 1. LOAD CKD-RELEVANT CODES FROM EXTERNAL FILE
# -----------------------------------------------------------------------------

import json
# Apply to your existing data
dx_cols = ["DX1", "DX2", "DX3", "DX4", "PDX"]  # Your diagnosis columns

def load_ckd_codes(json_file_path="ckd_codes.json"):
    """
    Load CKD-relevant codes from external JSON configuration file
    
    Args:
        json_file_path: Path to JSON file containing CKD codes
        
    Returns:
        tuple: (ckd_cpt_codes, ckd_comorbidity_codes)
    """
    try:
        with open(json_file_path, 'r') as f:
            codes_config = json.load(f)
        
        ckd_cpt_codes = codes_config["ckd_cpt_codes"]
        ckd_comorbidity_codes = codes_config["ckd_comorbidity_codes"]
        
        print(f"✅ Loaded {len(ckd_cpt_codes)} CPT codes and {len(ckd_comorbidity_codes)} comorbidity codes from {json_file_path}")
        return ckd_cpt_codes, ckd_comorbidity_codes
        
    except FileNotFoundError:
        print(f"❌ Could not find {json_file_path}. Using default code lists.")
        return get_default_ckd_codes()
    except json.JSONDecodeError:
        print(f"❌ Invalid JSON in {json_file_path}. Using default code lists.")
        return get_default_ckd_codes()

def get_default_ckd_codes():
    """Fallback default codes if JSON file not found"""
    default_cpt = [
        "90945", "90947", "90951", "90952", "82565", "82570", "84520",
        "99201", "99202", "99203", "99204", "99205", "99211", "99212", "99213", "99214", "99215"
    ]
    default_comorbidity = {
        "E119": "Type 2 diabetes mellitus without complications",
        "E1122": "Type 2 diabetes with diabetic CKD",
        "I10": "Essential hypertension", 
        "I120": "Hypertensive CKD with stage 1-4 CKD",
        "D631": "Anemia in CKD",
        "N179": "Acute kidney failure, unspecified"
    }
    return default_cpt, default_comorbidity

# Load the codes (will be used by other functions)
ckd_cpt_codes, ckd_comorbidity_codes = load_ckd_codes()
print(ckd_comorbidity_codes)

# -----------------------------------------------------------------------------
# 2. CKD STAGE DETECTION (Use existing robust function)
# -----------------------------------------------------------------------------

def ckd_stage_expr(dx_cols):
    """
    Returns a single Spark expression that gives the CKD stage as:
      1-6  … exact stage found
      0    … N189 (unspecified CKD)
      null … no CKD code in any DX column
    
    This function uses regex to handle CKD subcodes automatically
    and prioritizes diagnosis columns from left to right.
    """
    pieces = []
    for c in dx_cols:
        pieces.append(
            F.when(F.col(c) == "N189", F.lit(0))
            .otherwise(
                F.expr(f"try_cast(regexp_extract({c}, 'N18([1-6]).*', 1) AS int)")
            )
        )
    # coalesce from left to right → first non-null wins
    return F.coalesce(*pieces)


✅ Loaded 46 CPT codes and 83 comorbidity codes from ckd_codes.json
{'E1021': 'Type 1 diabetes with diabetic nephropathy', 'E1022': 'Type 1 diabetes with diabetic CKD', 'E1029': 'Type 1 diabetes with unspecified diabetic kidney complication', 'E1121': 'Type 2 diabetes with diabetic nephropathy', 'E1122': 'Type 2 diabetes with diabetic CKD', 'E1129': 'Type 2 diabetes with unspecified diabetic kidney complication', 'E1321': 'Other diabetes with diabetic nephropathy', 'E1322': 'Other diabetes with diabetic CKD', 'E1329': 'Other diabetes with unspecified diabetic kidney complication', 'E1140': 'Type 2 diabetes with diabetic neuropathy, unspecified', 'E1142': 'Type 2 diabetes with diabetic polyneuropathy', 'E1165': 'Type 2 diabetes with hyperglycemia', 'E119': 'Type 2 diabetes mellitus without complications', 'E109': 'Type 1 diabetes mellitus without complications', 'E139': 'Other specified diabetes mellitus without complications', 'I120': 'Hypertensive CKD with stage 1 through 4 CKD, or uns

In [5]:

# -----------------------------------------------------------------------------
# 3. FILTER TO CKD-RELEVANT CLAIMS ONLY
# -----------------------------------------------------------------------------
def filter_ckd_relevant_claims(claims_df, dx_cols, include_comorbidities=True):
    """
    Filter claims to CKD-relevant claims with multiple strategies:
    
    Strategy 1: Direct CKD diagnosis (N18x codes)
    Strategy 2: CKD-specific procedures (dialysis, etc.)
    Strategy 3: CKD comorbidities + ANY kidney-related procedure (expanded)
    
    Args:
        include_comorbidities: If True, includes claims with CKD comorbidities 
                              AND kidney-related procedures (more comprehensive)
    """
    
    # Condition 1: Direct CKD diagnosis using existing function
    ckd_dx_condition = ckd_stage_expr(dx_cols).isNotNull()
    
    # Condition 2: CKD-specific procedures (regardless of diagnosis)
    ckd_proc_condition = F.col("PROC1").isin(ckd_cpt_codes)
    
    if include_comorbidities:
        # Condition 3: CKD comorbidity diagnosis
        comorbidity_dx_condition = F.expr(" OR ".join([
            f"{col} = '{code}'" for col in dx_cols for code in ckd_comorbidity_codes.keys()
        ]))
        
        # Get claims that have comorbidity diagnosis but NOT direct CKD diagnosis
        # This ensures we're only adding new claims not already captured
        comorbidity_only_condition = comorbidity_dx_condition & ~ckd_dx_condition
        
        # Final condition: Direct CKD OR CKD procedure OR (comorbidity without direct CKD)
        final_condition = ckd_dx_condition | ckd_proc_condition | comorbidity_only_condition
        
        # For claim type classification - use "ckd_claim_type" to avoid conflict
        return claims_df.filter(final_condition).withColumn(
            "has_ckd_dx", ckd_dx_condition
        ).withColumn(
            "has_ckd_proc", ckd_proc_condition
        ).withColumn(
            "has_comorbidity", comorbidity_dx_condition
        ).withColumn(
            "ckd_claim_type",  # Changed from "claim_type" to "ckd_claim_type"
            F.when(F.col("has_ckd_dx"), "Direct_CKD")
            .when(F.col("has_ckd_proc") & ~F.col("has_ckd_dx"), "CKD_Procedure_Only") 
            .when(F.col("has_comorbidity") & ~F.col("has_ckd_dx"), "Comorbidity_CKD_Care")
            .otherwise("Other")
        )
    else:
        # Conservative approach: Only direct CKD diagnosis or CKD procedures
        return claims_df.filter(ckd_dx_condition | ckd_proc_condition).withColumn(
            "has_ckd_dx", ckd_dx_condition
        ).withColumn(
            "has_ckd_proc", ckd_proc_condition
        ).withColumn(
            "has_comorbidity", F.lit(False)
        ).withColumn(
            "ckd_claim_type",  # Changed from "claim_type" to "ckd_claim_type"
            F.when(F.col("has_ckd_dx"), "Direct_CKD")
            .when(F.col("has_ckd_proc"), "CKD_Procedure_Only")
            .otherwise("Other")
        )

In [6]:
def filter_ckd_relevant_claims_by_prefix(claims_df, dx_cols, include_comorbidities=True):
    """
    Filter CKD-relevant claims using code prefixes for maximum efficiency
    """
    # Define code prefixes
    ckd_specific_prefixes = [
        "N18",  # Direct CKD codes
        "E102", "E112", "E132",  # Diabetes with CKD
        "I12", "I13",  # Hypertensive CKD
        "Z94", "Z99"  # Transplant/dialysis status
    ]
    
    comorbidity_prefixes = [
        "E10", "E11", "E13",  # Diabetes without the specific CKD codes
        "I10", "I11", "I15",  # Hypertension without the specific CKD codes
        "N17", "N25",  # AKI and renal tubular disorders
        "D63", "D50", "D64",  # Anemia
        "E55", "E83", "E87",  # Mineral/electrolyte disorders
        "E78",  # Lipid disorders
        "I21", "I25", "I50",  # Cardiovascular conditions
        "R80",  # Proteinuria
        "Z79",  # Drug therapy
        "G47", "E03", "E66", "M10"  # Other comorbidities
    ]
    
    # Step 1: Direct CKD diagnosis filter
    ckd_dx_condition = ckd_stage_expr(dx_cols).isNotNull()
    
    # Step 2: CKD procedure filter
    ckd_proc_condition = F.col("PROC1").isin(ckd_cpt_codes)
    
    # Step 3: Create prefix-based filters
    specific_ckd_conditions = []
    for col in dx_cols:
        col_conditions = []
        for prefix in ckd_specific_prefixes:
            col_conditions.append(F.col(col).startswith(prefix))
        specific_ckd_conditions.append(F.array_contains(F.array(*col_conditions), F.lit(True)))
    specific_ckd_condition = F.array_contains(F.array(*specific_ckd_conditions), F.lit(True))
    
    # If including comorbidities, add those filters
    if include_comorbidities:
        comorbidity_conditions = []
        for col in dx_cols:
            col_conditions = []
            for prefix in comorbidity_prefixes:
                col_conditions.append(F.col(col).startswith(prefix))
            comorbidity_conditions.append(F.array_contains(F.array(*col_conditions), F.lit(True)))
        comorbidity_condition = F.array_contains(F.array(*comorbidity_conditions), F.lit(True))
        
        # Combine with procedure requirement
        comorbidity_with_proc_condition = comorbidity_condition & ckd_proc_condition & ~specific_ckd_condition
        
        # Final filter condition
        final_condition = specific_ckd_condition | ckd_proc_condition | comorbidity_with_proc_condition
    else:
        # Conservative approach: Only specific CKD or CKD procedures
        final_condition = specific_ckd_condition | ckd_proc_condition
    
    # Apply the filter
    filtered_claims = claims_df.filter(final_condition)
    
    # Add classification columns
    result = filtered_claims.withColumn(
        "has_ckd_dx", ckd_dx_condition
    ).withColumn(
        "has_ckd_proc", ckd_proc_condition
    )
    
    if include_comorbidities:
        result = result.withColumn(
            "has_comorbidity", comorbidity_condition
        )
    else:
        result = result.withColumn(
            "has_comorbidity", F.lit(False)
        )
    
    # Add claim type classification
    result = result.withColumn(
        "ckd_claim_type",
        F.when(F.col("has_ckd_dx"), "Direct_CKD")
         .when(F.col("has_ckd_proc") & ~F.col("has_ckd_dx"), "CKD_Procedure_Only")
         .when(F.col("has_comorbidity") & ~F.col("has_ckd_dx"), "Comorbidity_CKD_Care")
         .otherwise("Other")
    )
    
    return result

# Costs filtered with kidney-relevant procedural codes + including comorbidities costs

In [7]:
spark.conf.set("spark.sql.codegen.wholeStage", False)


In [8]:
def aggregate_ckd_costs_by_quarters(pilot_claims_df, dx_cols, target_year=2017, include_comorbidities=True, show_analysis=True):
    """
    Create CKD cost features from target_year data for pilot cohort
    
    Args:
        pilot_claims_df: Claims data already filtered to pilot cohort enrollees
        dx_cols: List of diagnosis columns to check
        target_year: Year to extract features from (default 2017 for 2018 prediction)
        include_comorbidities: Whether to include comorbidity+procedure combinations
        show_analysis: Whether to display claim pattern analysis
        
    Returns:
        DataFrame with quarterly CKD cost features per ENROLID
    """
    
    # Step 1: Filter to target year and CKD-relevant claims
    ckd_claims = (
        pilot_claims_df
        .filter(F.col("YEAR") == target_year)  # Only target year for feature engineering
        .transform(lambda df: filter_ckd_relevant_claims(df, dx_cols, include_comorbidities))
    )
    
    # Step 2: Add CKD stage and quarter information 
    # Step 3: Apply inflation adjustment to normalize to 2017 dollars
    inflation_map = {
        2017: 1.0,
        2018: 1.0685946832951803,
        2019: 1.1371893665903605
    }
    
    infl_map = sum([[F.lit(year), F.lit(factor)] for year, factor in inflation_map.items()], [])
    inflation_expr = F.create_map(*infl_map)
    
    ckd_claims_adjusted = (
        ckd_claims
        .withColumn("CKD_STAGE", ckd_stage_expr(dx_cols))
        .withColumn("QUARTER", F.quarter("SVCDATE"))
        .withColumn("inflation_factor", inflation_expr[F.col("YEAR")])
        .withColumn("NETPAY_ADJUSTED", F.col("NETPAY") / F.col("inflation_factor"))
        .drop("inflation_factor")
    )
    
    
    # OPTIONAL: Show claim pattern analysis using the enhanced dataframe
    if show_analysis:
        print(f"=== CKD CLAIM PATTERN ANALYSIS ({target_year}) ===")
        
        # 1. Claim type breakdown - using ckd_claim_type instead of claim_type
        print("\n1. CLAIM TYPE BREAKDOWN (COST PER YEAR):")
        pattern_summary = ckd_claims_adjusted.groupBy("ckd_claim_type").agg(
            F.count("*").alias("claim_count"),
            F.countDistinct("ENROLID").alias("unique_patients"),
            F.mean("NETPAY_ADJUSTED").alias("avg_cost"),
            F.sum("NETPAY_ADJUSTED").alias("total_cost")
        ).orderBy(F.desc("total_cost"))
        pattern_summary.show(truncate=False)
        
        # 2. Overall summary
       # total_claims = ckd_claims_enhanced.count()
       # total_patients = ckd_claims_enhanced.select("ENROLID").distinct().count()
       # total_cost = ckd_claims_enhanced.agg(F.sum("NETPAY_ADJUSTED")).collect()[0][0] or 0
       # avg_cost_per_claim = total_cost/total_claims if total_claims > 0 else 0
       # avg_cost_per_patient = total_cost/total_patients if total_patients > 0 else 0
        
        #        print(f"   Total CKD-relevant claims: {ckd_claims_enhanced.count():,}")
        #       print(f"   Unique patients: {total_patients:,}")
        #      print(f"   Total cost: ${total_cost:,.2f}")
        #     print(f"   Average cost per claim: ${avg_cost_per_claim:.2f}")
        #    print(f"   Average cost per patient: ${avg_cost_per_patient:.2f}")
            
    

    # Step 4: Create quarterly CKD cost features 
    quarterly_features = (
        ckd_claims_adjusted
        .groupBy("ENROLID", "QUARTER")
        .agg(
            # Primary cost features
            F.sum(F.col("NETPAY_ADJUSTED")).alias("total_ckd_cost_3month"),
            F.count("*").alias("ckd_claim_count_3month"),
            
            # CKD stage features (only for direct CKD diagnosis claims)
            F.max(F.when(F.col("CKD_STAGE").isNotNull(), F.col("CKD_STAGE"))).alias("max_ckd_stage_3month"),
            F.mean(F.when(F.col("CKD_STAGE").isNotNull(), F.col("CKD_STAGE"))).alias("avg_ckd_stage_3month"),
            
            # Cost breakdown by claim type (for model interpretability)
            F.sum(F.when(F.col("ckd_claim_type") == "Direct_CKD", F.col("NETPAY_ADJUSTED")).otherwise(0)).alias("direct_ckd_cost_3month"),
            F.sum(F.when(F.col("ckd_claim_type") == "CKD_Procedure_Only", F.col("NETPAY_ADJUSTED")).otherwise(0)).alias("procedure_only_cost_3month"),
            F.sum(F.when(F.col("ckd_claim_type") == "Comorbidity_CKD_Care", F.col("NETPAY_ADJUSTED")).otherwise(0)).alias("comorbidity_ckd_cost_3month"),
            
            # Utilization features
            F.sum(F.when(F.col("ckd_claim_type") == "Direct_CKD", 1).otherwise(0)).alias("direct_ckd_claims_3month"),
            F.sum(F.when(F.col("ckd_claim_type") == "CKD_Procedure_Only", 1).otherwise(0)).alias("procedure_only_claims_3month"),
            F.sum(F.when(F.col("ckd_claim_type") == "Comorbidity_CKD_Care", 1).otherwise(0)).alias("comorbidity_ckd_claims_3month")
        )
        .withColumn("YEAR", F.lit(target_year))  # Add year back for pivot
    )
    
    return quarterly_features

In [9]:
# -----------------------------------------------------------------------------
# 5. CREATE WIDE FORMAT FOR ANALYSIS
# -----------------------------------------------------------------------------

def create_wide_format_features(quarterly_features_df, target_year=2017):
    """
    Convert quarterly CKD cost features to wide format for ML model
    
    Creates features like:
    - 2017Q1_ckd_cost, 2017Q2_ckd_cost, etc.
    - total_ckd_cost_2017 (annual sum)
    - ckd_cost_trend_2017 (Q4-Q1 difference)
    """
    
    quarters = [1, 2, 3, 4]
    quarter_labels = [f"{target_year}Q{q}" for q in quarters]
    
    # Pivot to wide format
    features_wide = (
        quarterly_features_df
        .withColumn("YQ", F.concat_ws("Q", F.col("YEAR"), F.col("QUARTER")))
        .groupBy("ENROLID")
        .pivot("YQ", quarter_labels)
        .agg(
            F.first("total_ckd_cost_3month").alias("ckd_cost"),
            F.first("ckd_claim_count_3month").alias("ckd_claims"),
            F.first("max_ckd_stage_3month").alias("max_stage"),
            F.first("direct_ckd_cost_3month").alias("direct_cost"),
            F.first("procedure_only_cost_3month").alias("proc_cost"),
            F.first("comorbidity_ckd_cost_3month").alias("comorbidity_cost")
        )
    )
    
    # Flatten column names and fill nulls
    for q in quarters:
        quarter_label = f"{target_year}Q{q}"
        features_wide = (features_wide
            .withColumnRenamed(f"{quarter_label}_ckd_cost", f"{quarter_label}_ckd_cost")
            .withColumnRenamed(f"{quarter_label}_ckd_claims", f"{quarter_label}_ckd_claims")
            .withColumnRenamed(f"{quarter_label}_max_stage", f"{quarter_label}_max_ckd_stage")
            .withColumnRenamed(f"{quarter_label}_direct_cost", f"{quarter_label}_direct_ckd_cost")
            .withColumnRenamed(f"{quarter_label}_proc_cost", f"{quarter_label}_procedure_ckd_cost")
            .withColumnRenamed(f"{quarter_label}_comorbidity_cost", f"{quarter_label}_comorbidity_ckd_cost")
        )
    
    # Fill nulls with zeros
    fill_values = {}
    for q in quarters:
        fill_values[f"{target_year}Q{q}_ckd_cost"] = 0.0
        fill_values[f"{target_year}Q{q}_ckd_claims"] = 0
        fill_values[f"{target_year}Q{q}_max_ckd_stage"] = 0
        fill_values[f"{target_year}Q{q}_direct_ckd_cost"] = 0.0
        fill_values[f"{target_year}Q{q}_procedure_ckd_cost"] = 0.0
        fill_values[f"{target_year}Q{q}_comorbidity_ckd_cost"] = 0.0
    
    features_filled = features_wide.fillna(fill_values)
    
    # Add derived annual features
    cost_columns = [f"{target_year}Q{q}_ckd_cost" for q in quarters]
    
    features_final = features_filled.withColumn(
        f"total_ckd_cost_{target_year}",
        sum([F.coalesce(F.col(col), F.lit(0.0)) for col in cost_columns])
    ).withColumn(
        f"ckd_cost_trend_{target_year}",
        F.coalesce(F.col(f"{target_year}Q4_ckd_cost"), F.lit(0.0)) - 
        F.coalesce(F.col(f"{target_year}Q1_ckd_cost"), F.lit(0.0))
    ).withColumn(
        f"ckd_cost_volatility_{target_year}",
        F.sqrt(
            sum([(F.coalesce(F.col(f"{target_year}Q{q}_ckd_cost"), F.lit(0.0)) - 
                  F.col(f"total_ckd_cost_{target_year}") / 4) ** 2 for q in quarters]) / 4
        )
    )
    
    return features_final



In [11]:
year=2017
sample_fraction = 0.1
print(f"DEBUG MODE: Using {sample_fraction*100:.2f}% of data")
#debug_claims = pilot_claims.sample(withReplacement=False, fraction=sample_fraction)
debug_claims = pilot_claims


DEBUG MODE: Using 10.00% of data


In [12]:
# Approach 2: Comprehensive (includes comorbidity + procedure combinations)  
comprehensive_costs = aggregate_ckd_costs_by_quarters(debug_claims, dx_cols, year, include_comorbidities=True, show_analysis=False)

In [41]:
# Show breakdown of comprehensive approach
print("\n=== COMPREHENSIVE APPROACH BREAKDOWN ===")
cost_breakdown = comprehensive_costs.agg(
    F.sum("direct_ckd_cost_3month").alias("direct_ckd_total"),
    F.sum("procedure_only_cost_3month").alias("procedure_only_total"), 
    F.sum("comorbidity_ckd_cost_3month").alias("comorbidity_ckd_total")
).collect()[0]

print(f"Direct CKD diagnosis costs: ${cost_breakdown['direct_ckd_total'] or 0:,.2f}")
print(f"CKD procedure only costs: ${cost_breakdown['procedure_only_total'] or 0:,.2f}")
print(f"Comorbidity + CKD care costs: ${cost_breakdown['comorbidity_ckd_total'] or 0:,.2f}")
# Get total directly
comprehensive_total = comprehensive_costs.agg(F.sum("total_ckd_cost_3month")).collect()[0][0] or 0

# Calculate total from breakdown for verification
breakdown_total = (
    cost_breakdown['direct_ckd_total'] + 
    cost_breakdown['procedure_only_total'] + 
    cost_breakdown['comorbidity_ckd_total']
)

print(f"Direct verification: {comprehensive_total:,.2f}")
print(f"From breakdown sum: {breakdown_total:,.2f}")


=== COMPREHENSIVE APPROACH BREAKDOWN ===


25/08/16 15:58:32 ERROR CodeGenerator: Failed to compile the generated Java code.
org.codehaus.commons.compiler.InternalCompilerException: Compiling "GeneratedClass" in File 'generated.java', Line 1, Column 1: File 'generated.java', Line 299, Column 14: Compiling "hashAgg_doAggregateWithKeys_0()"
	at org.codehaus.janino.UnitCompiler.compile2(UnitCompiler.java:402)
	at org.codehaus.janino.UnitCompiler.access$000(UnitCompiler.java:236)
	at org.codehaus.janino.UnitCompiler$2.visitCompilationUnit(UnitCompiler.java:363)
	at org.codehaus.janino.UnitCompiler$2.visitCompilationUnit(UnitCompiler.java:361)
	at org.codehaus.janino.Java$CompilationUnit.accept(Java.java:371)
	at org.codehaus.janino.UnitCompiler.compileUnit(UnitCompiler.java:361)
	at org.codehaus.janino.SimpleCompiler.cook(SimpleCompiler.java:264)
	at org.codehaus.janino.ClassBodyEvaluator.cook(ClassBodyEvaluator.java:294)
	at org.codehaus.janino.ClassBodyEvaluator.cook(ClassBodyEvaluator.java:288)
	at org.codehaus.janino.ClassBodyE

Direct CKD diagnosis costs: $11,033,786.16
CKD procedure only costs: $2,303,368.22
Comorbidity + CKD care costs: $34,069,703.12


Direct verification: 48,182,228.59
From breakdown sum: 47,406,857.50


In [13]:

# Step 3: Choose your preferred approach and get final results
print("\n=== STEP 3: FINAL CKD COST AGGREGATION ===")
# Recommended: Use comprehensive approach unless it includes too much noise
quarterly_ckd_costs = comprehensive_costs

# Show sample results
#print("Sample quarterly CKD costs:")
#quarterly_ckd_costs.show(10)
# Step 4: Create wide format for analysis
wide_ckd_costs = create_wide_format_features(quarterly_ckd_costs, target_year=2017)
print("\nWide format sample:")
wide_ckd_costs.show(5)

# Step 5: Summary statistics
print("\nQuarterly summary:")
quarterly_ckd_costs.groupBy("QUARTER").agg(
    F.count("ENROLID").alias("enrollees_with_ckd_costs"),
    F.mean("total_ckd_cost_3month").alias("avg_quarterly_cost"),
    F.sum("total_ckd_cost_3month").alias("total_quarterly_cost"),
    F.mean("ckd_claim_count_3month").alias("avg_claims_per_enrollee")
).orderBy("QUARTER").show()




=== STEP 3: FINAL CKD COST AGGREGATION ===

Wide format sample:


+-----------+------------------+-----------------+--------------------+----------------------+-------------------------+---------------------------+-----------------+-----------------+--------------------+----------------------+-------------------------+---------------------------+------------------+-----------------+--------------------+----------------------+-------------------------+---------------------------+------------------+-----------------+--------------------+----------------------+-------------------------+---------------------------+-------------------+-------------------+------------------------+
|    ENROLID|   2017Q1_ckd_cost|2017Q1_ckd_claims|2017Q1_max_ckd_stage|2017Q1_direct_ckd_cost|2017Q1_procedure_ckd_cost|2017Q1_comorbidity_ckd_cost|  2017Q2_ckd_cost|2017Q2_ckd_claims|2017Q2_max_ckd_stage|2017Q2_direct_ckd_cost|2017Q2_procedure_ckd_cost|2017Q2_comorbidity_ckd_cost|   2017Q3_ckd_cost|2017Q3_ckd_claims|2017Q3_max_ckd_stage|2017Q3_direct_ckd_cost|2017Q3_procedure_ck

+-------+------------------------+------------------+--------------------+-----------------------+
|QUARTER|enrollees_with_ckd_costs|avg_quarterly_cost|total_quarterly_cost|avg_claims_per_enrollee|
+-------+------------------------+------------------+--------------------+-----------------------+
|      1|                   28453|3582.7043239728578|1.0193868612999973E8|     19.653779917759113|
|      2|                   28966|3952.2862404197876|1.1448192323999956E8|     19.951080577228474|
|      3|                   29442| 4374.838038176758|1.2880398152000012E8|     20.792609197744717|
|      4|                   30124| 4417.495529810114|1.3307263533999987E8|     21.390353206745452|
+-------+------------------------+------------------+--------------------+-----------------------+



In [14]:
wide_ckd_costs.select("ENROLID").distinct().count()


33590

Run the comparison function first to see how much additional cost is captured. If it seems reasonable (not inflating costs by 200%+), use the comprehensive approach for more complete CKD cost accounting.

# Track cost derivatives and temporal stats quarterly

In [4]:
# -----------------------------------------------------------------------------
# QUARTERLY DERIVATIVES AND STATISTICAL FEATURES FOR CKD COSTS
# -----------------------------------------------------------------------------
# This extends the existing wide format CKD features with temporal patterns

def add_quarterly_derivatives(ckd_features_wide, target_year=2017):
    """
    Add quarterly derivative features to existing wide format CKD cost features
    
    Args:
        ckd_features_wide: Output from create_wide_format_features()
        target_year: Year for feature naming
        
    Returns:
        DataFrame with additional quarterly derivative features
    """
    
    print(f"=== ADDING QUARTERLY DERIVATIVES FOR {target_year} ===")
    
    # Define quarterly cost columns
    q_cols = [f"{target_year}Q{q}_ckd_cost" for q in [1, 2, 3, 4]]
    
    # 1. QUARTERLY DERIVATIVES (Q-to-Q changes)
    derivatives_df = ckd_features_wide.withColumn(
        f"ckd_cost_deriv_Q1_Q2_{target_year}",
        F.coalesce(F.col(f"{target_year}Q2_ckd_cost"), F.lit(0.0)) - 
        F.coalesce(F.col(f"{target_year}Q1_ckd_cost"), F.lit(0.0))
    ).withColumn(
        f"ckd_cost_deriv_Q2_Q3_{target_year}",
        F.coalesce(F.col(f"{target_year}Q3_ckd_cost"), F.lit(0.0)) - 
        F.coalesce(F.col(f"{target_year}Q2_ckd_cost"), F.lit(0.0))
    ).withColumn(
        f"ckd_cost_deriv_Q3_Q4_{target_year}",
        F.coalesce(F.col(f"{target_year}Q4_ckd_cost"), F.lit(0.0)) - 
        F.coalesce(F.col(f"{target_year}Q3_ckd_cost"), F.lit(0.0))
    )
    
    # 2. BINARY INCREASING FLAGS
    derivatives_df = derivatives_df.withColumn(
        f"is_increasing_Q1_Q2_{target_year}",
        F.when(F.col(f"ckd_cost_deriv_Q1_Q2_{target_year}") > 0, 1).otherwise(0)
    ).withColumn(
        f"is_increasing_Q2_Q3_{target_year}",
        F.when(F.col(f"ckd_cost_deriv_Q2_Q3_{target_year}") > 0, 1).otherwise(0)
    ).withColumn(
        f"is_increasing_Q3_Q4_{target_year}",
        F.when(F.col(f"ckd_cost_deriv_Q3_Q4_{target_year}") > 0, 1).otherwise(0)
    )
    
    # 3. AGGREGATED DERIVATIVE PATTERNS
    derivatives_df = derivatives_df.withColumn(
        f"total_increasing_quarters_{target_year}",
        F.col(f"is_increasing_Q1_Q2_{target_year}") + 
        F.col(f"is_increasing_Q2_Q3_{target_year}") + 
        F.col(f"is_increasing_Q3_Q4_{target_year}")
    ).withColumn(
        f"is_consistently_increasing_{target_year}",
        F.when(F.col(f"total_increasing_quarters_{target_year}") == 3, 1).otherwise(0)
    ).withColumn(
        f"is_consistently_decreasing_{target_year}",
        F.when(F.col(f"total_increasing_quarters_{target_year}") == 0, 1).otherwise(0)
    )
    
    # 4. AVERAGE QUARTERLY DERIVATIVE (rate of change)
    derivatives_df = derivatives_df.withColumn(
        f"avg_quarterly_derivative_{target_year}",
        (F.col(f"ckd_cost_deriv_Q1_Q2_{target_year}") + 
         F.col(f"ckd_cost_deriv_Q2_Q3_{target_year}") + 
         F.col(f"ckd_cost_deriv_Q3_Q4_{target_year}")) / 3
    )
    
    print(f"✅ Added quarterly derivative features")
    return derivatives_df


def add_temporal_statistical_features(ckd_features_wide, target_year=2017):
    """
    Add skewness, kurtosis, and other statistical measures of quarterly cost distribution
    
    Args:
        ckd_features_wide: Output from create_wide_format_features() or add_quarterly_derivatives()
        target_year: Year for feature naming
        
    Returns:
        DataFrame with additional statistical features
    """
    
    print(f"=== ADDING TEMPORAL STATISTICAL FEATURES FOR {target_year} ===")
    
    # Define quarterly cost columns 
    q_cols = [f"{target_year}Q{q}_ckd_cost" for q in [1, 2, 3, 4]]
    
    # Helper: coalesce all quarterly costs to handle nulls
    q1 = F.coalesce(F.col(f"{target_year}Q1_ckd_cost"), F.lit(0.0))
    q2 = F.coalesce(F.col(f"{target_year}Q2_ckd_cost"), F.lit(0.0))
    q3 = F.coalesce(F.col(f"{target_year}Q3_ckd_cost"), F.lit(0.0))
    q4 = F.coalesce(F.col(f"{target_year}Q4_ckd_cost"), F.lit(0.0))
    
    # Calculate quarterly mean and variance for skew/kurtosis
    stats_df = ckd_features_wide.withColumn(
        f"quarterly_mean_{target_year}",
        (q1 + q2 + q3 + q4) / 4
    ).withColumn(
        f"quarterly_variance_{target_year}",
        ((q1 - F.col(f"quarterly_mean_{target_year}")) ** 2 +
         (q2 - F.col(f"quarterly_mean_{target_year}")) ** 2 +
         (q3 - F.col(f"quarterly_mean_{target_year}")) ** 2 +
         (q4 - F.col(f"quarterly_mean_{target_year}")) ** 2) / 4
    ).withColumn(
        f"quarterly_std_{target_year}",
        F.sqrt(F.col(f"quarterly_variance_{target_year}"))
    )
    
    # 1. SKEWNESS (asymmetry of cost distribution)
    # Positive skew = costs weighted toward later quarters
    # Negative skew = costs weighted toward earlier quarters
    stats_df = stats_df.withColumn(
        f"quarterly_skewness_{target_year}",
        F.when(F.col(f"quarterly_std_{target_year}") > 0,
            (((q1 - F.col(f"quarterly_mean_{target_year}")) / F.col(f"quarterly_std_{target_year}")) ** 3 +
             ((q2 - F.col(f"quarterly_mean_{target_year}")) / F.col(f"quarterly_std_{target_year}")) ** 3 +
             ((q3 - F.col(f"quarterly_mean_{target_year}")) / F.col(f"quarterly_std_{target_year}")) ** 3 +
             ((q4 - F.col(f"quarterly_mean_{target_year}")) / F.col(f"quarterly_std_{target_year}")) ** 3) / 4
        ).otherwise(0.0)
    )
    
    # 2. KURTOSIS (tail heaviness of cost distribution)
    # High kurtosis = extreme quarterly cost spikes
    # Low kurtosis = consistent quarterly costs
    stats_df = stats_df.withColumn(
        f"quarterly_kurtosis_{target_year}",
        F.when(F.col(f"quarterly_std_{target_year}") > 0,
            (((q1 - F.col(f"quarterly_mean_{target_year}")) / F.col(f"quarterly_std_{target_year}")) ** 4 +
             ((q2 - F.col(f"quarterly_mean_{target_year}")) / F.col(f"quarterly_std_{target_year}")) ** 4 +
             ((q3 - F.col(f"quarterly_mean_{target_year}")) / F.col(f"quarterly_std_{target_year}")) ** 4 +
             ((q4 - F.col(f"quarterly_mean_{target_year}")) / F.col(f"quarterly_std_{target_year}")) ** 4) / 4 - 3
        ).otherwise(0.0)
    )
    
    # 3. COEFFICIENT OF VARIATION (relative variability)
    # High CV = volatile costs relative to mean
    stats_df = stats_df.withColumn(
        f"quarterly_cv_{target_year}",
        F.when(F.col(f"quarterly_mean_{target_year}") > 0,
            F.col(f"quarterly_std_{target_year}") / F.col(f"quarterly_mean_{target_year}")
        ).otherwise(0.0)
    )
    
    # 4. RANGE FEATURES
    stats_df = stats_df.withColumn(
        f"quarterly_max_{target_year}",
        F.greatest(q1, q2, q3, q4)
    ).withColumn(
        f"quarterly_min_{target_year}",
        F.least(q1, q2, q3, q4)
    ).withColumn(
        f"quarterly_range_{target_year}",
        F.col(f"quarterly_max_{target_year}") - F.col(f"quarterly_min_{target_year}")
    )
    
    # 5. INTERPRETABLE CATEGORICAL FEATURES
    stats_df = stats_df.withColumn(
        f"cost_pattern_{target_year}",
        F.when(F.col(f"quarterly_skewness_{target_year}") > 0.5, "late_heavy")
        .when(F.col(f"quarterly_skewness_{target_year}") < -0.5, "early_heavy")
        .otherwise("balanced")
    ).withColumn(
        f"cost_stability_{target_year}",
        F.when(F.col(f"quarterly_cv_{target_year}") < 0.3, "stable")
        .when(F.col(f"quarterly_cv_{target_year}") > 1.0, "highly_volatile")
        .otherwise("moderate")
    )
    
    # Clean up intermediate columns
    stats_df = stats_df.drop("quarterly_mean_" + str(target_year), 
                            "quarterly_variance_" + str(target_year))
    
    print(f"✅ Added temporal statistical features")
    return stats_df


wide_ckd_costs = spark.read.format("parquet").load( "0813_cost_features_2017.parquet")

# Add derivatives directly to your wide format data
derivatives_df = add_quarterly_derivatives(wide_ckd_costs, target_year=2017)

# Then add statistical features
enhanced_features = add_temporal_statistical_features(derivatives_df, target_year=2017)

# Show some sample results
enhanced_features.select(
    "ENROLID", 
    "ckd_cost_deriv_Q1_Q2_2017", "ckd_cost_deriv_Q2_Q3_2017", "ckd_cost_deriv_Q3_Q4_2017",
    "is_consistently_increasing_2017", "is_consistently_decreasing_2017", 
    "quarterly_skewness_2017", "cost_pattern_2017", "cost_stability_2017"
).show(5)

=== ADDING QUARTERLY DERIVATIVES FOR 2017 ===
✅ Added quarterly derivative features
=== ADDING TEMPORAL STATISTICAL FEATURES FOR 2017 ===
✅ Added temporal statistical features
+-----------+-------------------------+-------------------------+-------------------------+-------------------------------+-------------------------------+-----------------------+-----------------+-------------------+
|    ENROLID|ckd_cost_deriv_Q1_Q2_2017|ckd_cost_deriv_Q2_Q3_2017|ckd_cost_deriv_Q3_Q4_2017|is_consistently_increasing_2017|is_consistently_decreasing_2017|quarterly_skewness_2017|cost_pattern_2017|cost_stability_2017|
+-----------+-------------------------+-------------------------+-------------------------+-------------------------------+-------------------------------+-----------------------+-----------------+-------------------+
| 3135505001|                  -419.85|                   110.28|                    -25.0|                              0|                              0|     0.92016793

In [7]:
enhanced_features.columns

['ENROLID',
 'AGEGRP',
 'SEX',
 'REGION',
 'INDSTRY',
 'has_Hypertension',
 'has_Type_2_Diabetes',
 'has_Anemia',
 'has_Hyperlipidemia',
 'has_Acute_Kidney_Failure',
 'has_Hyperparathyroidism',
 'has_Kidney_Transplant',
 'has_Vitamin_D_Deficiency',
 'has_Long-term_Drug_Therapy',
 'has_Hypothyroidism',
 'has_Sleep_Apnea',
 '2017Q1',
 '2017Q2',
 '2017Q3',
 '2017Q4',
 'stage_2017',
 'cost_stratum_2017',
 'cost_stratum_2018',
 'util_2017',
 'THRCLS_53',
 'THRCLS_51',
 'THRCLS_52',
 'THRCLS_69',
 'THRCLS_46',
 'THRCLS_47',
 'THRCLS_172',
 'THRCLS_174',
 'THRCLS_181',
 'THRCLS_60',
 '2017Q1_ckd_cost',
 '2017Q1_ckd_claims',
 '2017Q1_max_ckd_stage',
 '2017Q1_direct_ckd_cost',
 '2017Q1_procedure_ckd_cost',
 '2017Q1_comorbidity_ckd_cost',
 '2017Q2_ckd_cost',
 '2017Q2_ckd_claims',
 '2017Q2_max_ckd_stage',
 '2017Q2_direct_ckd_cost',
 '2017Q2_procedure_ckd_cost',
 '2017Q2_comorbidity_ckd_cost',
 '2017Q3_ckd_cost',
 '2017Q3_ckd_claims',
 '2017Q3_max_ckd_stage',
 '2017Q3_direct_ckd_cost',
 '2017Q3_pr

In [8]:
enhanced_features.write.mode("overwrite").parquet("0813_cost_with_derivatives_2017.parquet")


## CLINICAL INTERPRETATION OF TEMPORAL CKD FEATURES


In [50]:

def interpret_ckd_temporal_patterns(enhanced_features_df, target_year=2017, show_examples=True):
    """
    Analyze and interpret the temporal patterns in CKD costs for clinical insights
    
    Args:
        enhanced_features_df: Output from create_enhanced_ckd_features()
        target_year: Year being analyzed
        show_examples: Whether to show example patients
    """
    
    print(f"=== CLINICAL INTERPRETATION OF {target_year} CKD TEMPORAL PATTERNS ===")
    
    # Filter to patients with CKD costs
    patients_with_costs = enhanced_features_df.filter(F.col(f"total_ckd_cost_{target_year}") > 0)
    
    # 1. DERIVATIVE PATTERNS ANALYSIS
    print("\n1. DISEASE PROGRESSION PATTERNS (Quarterly Derivatives):")
    
    progression_summary = patients_with_costs.agg(
        F.mean(f"is_consistently_increasing_{target_year}").alias("pct_consistently_increasing"),
        F.mean(f"is_consistently_decreasing_{target_year}").alias("pct_consistently_decreasing"),
        F.mean(f"total_increasing_quarters_{target_year}").alias("avg_increasing_quarters"),
        F.mean(f"avg_quarterly_derivative_{target_year}").alias("avg_rate_of_change")
    ).collect()[0]
    
    print(f"   • {progression_summary['pct_consistently_increasing']*100:.1f}% of patients have consistently INCREASING costs (worsening)")
    print(f"   • {progression_summary['pct_consistently_decreasing']*100:.1f}% of patients have consistently DECREASING costs (improving)")
    print(f"   • Average quarters with increasing costs: {progression_summary['avg_increasing_quarters']:.1f}/3")
    print(f"   • Average quarterly cost change: ${progression_summary['avg_rate_of_change']:.2f}")
    
    # 2. STATISTICAL PATTERNS ANALYSIS  
    print(f"\n2. TEMPORAL DISTRIBUTION PATTERNS:")
    
    pattern_summary = patients_with_costs.groupBy(f"cost_pattern_{target_year}").agg(
        F.count("*").alias("patient_count"),
        F.mean(f"total_ckd_cost_{target_year}").alias("avg_total_cost")
    ).orderBy("patient_count")
    
    print("   Cost timing patterns:")
    pattern_summary.show(truncate=False)
    
    stability_summary = patients_with_costs.groupBy(f"cost_stability_{target_year}").agg(
        F.count("*").alias("patient_count"),
        F.mean(f"quarterly_cv_{target_year}").alias("avg_coefficient_variation")
    ).orderBy("patient_count")
    
    print("   Cost stability patterns:")
    stability_summary.show(truncate=False)
    
    # 3. CLINICAL INSIGHTS
    print(f"\n3. CLINICAL INSIGHTS:")
    
    # High-risk patterns for 2018
    high_risk_increasing = patients_with_costs.filter(F.col(f"is_consistently_increasing_{target_year}") == 1).count()
    volatile_patients = patients_with_costs.filter(F.col(f"cost_stability_{target_year}") == "highly_volatile").count()
    late_heavy_patients = patients_with_costs.filter(F.col(f"cost_pattern_{target_year}") == "late_heavy").count()
    
    total_patients = patients_with_costs.count()
    
    print(f"   🚨 HIGH RISK FOR 2018 (consistently increasing): {high_risk_increasing} patients ({high_risk_increasing/total_patients*100:.1f}%)")
    print(f"   ⚠️  VOLATILE PATTERNS (unpredictable spikes): {volatile_patients} patients ({volatile_patients/total_patients*100:.1f}%)")
    print(f"   📈 LATE-YEAR ESCALATION (Q4 heavy costs): {late_heavy_patients} patients ({late_heavy_patients/total_patients*100:.1f}%)")
    
    # 4. EXAMPLE PATIENTS
    if show_examples:
        print(f"\n4. EXAMPLE PATIENT PATTERNS:")
        
        # Example 1: Consistently worsening
        worsening_example = patients_with_costs.filter(
            F.col(f"is_consistently_increasing_{target_year}") == 1
        ).select(
            "ENROLID",
            f"{target_year}Q1_ckd_cost", f"{target_year}Q2_ckd_cost", 
            f"{target_year}Q3_ckd_cost", f"{target_year}Q4_ckd_cost",
            f"ckd_cost_deriv_Q1_Q2_{target_year}", f"ckd_cost_deriv_Q2_Q3_{target_year}", f"ckd_cost_deriv_Q3_Q4_{target_year}"
        ).first()
        
        if worsening_example:
            print("\n   WORSENING PATIENT EXAMPLE:")
            print(f"   ENROLID: {worsening_example['ENROLID']}")
            print(f"   Quarterly costs: Q1=${worsening_example[f'{target_year}Q1_ckd_cost']:.0f} → Q2=${worsening_example[f'{target_year}Q2_ckd_cost']:.0f} → Q3=${worsening_example[f'{target_year}Q3_ckd_cost']:.0f} → Q4=${worsening_example[f'{target_year}Q4_ckd_cost']:.0f}")
            print(f"   Quarterly changes: Q1→Q2: +${worsening_example[f'ckd_cost_deriv_Q1_Q2_{target_year}']:.0f}, Q2→Q3: +${worsening_example[f'ckd_cost_deriv_Q2_Q3_{target_year}']:.0f}, Q3→Q4: +${worsening_example[f'ckd_cost_deriv_Q3_Q4_{target_year}']:.0f}")
            print("   → Clinical insight: Disease progression, likely needs intervention")
        
        # Example 2: Highly volatile
        volatile_example = patients_with_costs.filter(
            F.col(f"cost_stability_{target_year}") == "highly_volatile"
        ).select(
            "ENROLID",
            f"{target_year}Q1_ckd_cost", f"{target_year}Q2_ckd_cost", 
            f"{target_year}Q3_ckd_cost", f"{target_year}Q4_ckd_cost",
            f"quarterly_cv_{target_year}"
        ).first()
        
        if volatile_example:
            print(f"\n   VOLATILE PATIENT EXAMPLE:")
            print(f"   ENROLID: {volatile_example['ENROLID']}")
            print(f"   Quarterly costs: Q1=${volatile_example[f'{target_year}Q1_ckd_cost']:.0f}, Q2=${volatile_example[f'{target_year}Q2_ckd_cost']:.0f}, Q3=${volatile_example[f'{target_year}Q3_ckd_cost']:.0f}, Q4=${volatile_example[f'{target_year}Q4_ckd_cost']:.0f}")
            print(f"   Coefficient of variation: {volatile_example[f'quarterly_cv_{target_year}']:.2f}")
            print("   → Clinical insight: Unpredictable complications, high risk for sudden spikes")

interpret_ckd_temporal_patterns(enhanced_features, 2017, show_examples=True)


=== CLINICAL INTERPRETATION OF 2017 CKD TEMPORAL PATTERNS ===

1. DISEASE PROGRESSION PATTERNS (Quarterly Derivatives):


   • 2.1% of patients have consistently INCREASING costs (worsening)
   • 9.3% of patients have consistently DECREASING costs (improving)
   • Average quarters with increasing costs: 1.2/3
   • Average quarterly cost change: $22.09

2. TEMPORAL DISTRIBUTION PATTERNS:
   Cost timing patterns:


+-----------------+-------------+------------------+
|cost_pattern_2017|patient_count|avg_total_cost    |
+-----------------+-------------+------------------+
|early_heavy      |589          |2035.9665534804758|
|balanced         |5788         |1500.5781219764983|
|late_heavy       |20353        |2030.892129415816 |
+-----------------+-------------+------------------+

   Cost stability patterns:


+-------------------+-------------+-------------------------+
|cost_stability_2017|patient_count|avg_coefficient_variation|
+-------------------+-------------+-------------------------+
|stable             |135          |0.2188572044383514       |
|moderate           |4964         |0.7693941200386076       |
|highly_volatile    |21670        |1.510949760337427        |
+-------------------+-------------+-------------------------+


3. CLINICAL INSIGHTS:


   🚨 HIGH RISK FOR 2018 (consistently increasing): 592 patients (2.2%)
   ⚠️  VOLATILE PATTERNS (unpredictable spikes): 21604 patients (80.8%)
   📈 LATE-YEAR ESCALATION (Q4 heavy costs): 20298 patients (75.9%)

4. EXAMPLE PATIENT PATTERNS:



   WORSENING PATIENT EXAMPLE:
   ENROLID: 3144387101
   Quarterly costs: Q1=$17 → Q2=$22 → Q3=$81 → Q4=$87
   Quarterly changes: Q1→Q2: +$5, Q2→Q3: +$59, Q3→Q4: +$6
   → Clinical insight: Disease progression, likely needs intervention



   VOLATILE PATIENT EXAMPLE:
   ENROLID: 1124460101
   Quarterly costs: Q1=$0, Q2=$170, Q3=$0, Q4=$0
   Coefficient of variation: 1.73
   → Clinical insight: Unpredictable complications, high risk for sudden spikes


### If doing 2018 instead 2017 data, careful with column names!

In [11]:
# 1. Define old and new column names
year = 2017
old_cols = [f"{y}Q{q}" for y in year for q in [1, 2, 3, 4]]
new_cols = [f"{y-1}Q{q}" for y in year for q in [1, 2, 3, 4]]

# 2. Create a renamed version of df_stage_pivot
for old, new in zip(old_cols, new_cols):
    df_stage_pivot = df_stage_pivot.withColumnRenamed(old, new)

# Optional: verify
df_stage_pivot.show(5)

+-----------+------+------+------+------+----------+
|    ENROLID|2017Q1|2017Q2|2017Q3|2017Q4|stage_2017|
+-----------+------+------+------+------+----------+
| 2687114401|     4|     6|     5|     4|         4|
|33142058701|     3|     3|     3|    -1|         3|
| 1469126402|     3|     3|     3|     3|         3|
| 3062436801|     3|     3|     3|     2|         2|
| 4113918701|    -1|     3|     3|     3|         3|
+-----------+------+------+------+------+----------+
only showing top 5 rows
